# Agrupación en clústeres — K-Means y jerárquica

En el cuaderno anterior vimos que los datos se pueden agrupar y cómo saber si son compatibles con ese análisis. Aquí hacemos la agrupación de forma automática, con **dos algoritmos distintos**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

data = pd.read_csv('../datasets/seeds.csv')
features = data[data.columns[0:6]]

# Normalizar y reducir a 2D para poder visualizar (igual que en el cuaderno 01)
scaled_features = MinMaxScaler().fit_transform(features)
pca = PCA(n_components=2, random_state=0).fit(scaled_features)
features_2d = pca.transform(scaled_features)

print(f'{len(features)} observaciones · {features.shape[1]} features → proyectadas a 2D')

## K-Means

Separa el dataset en *K* clústeres de varianza similar. El número de clústeres lo define el usuario. Los pasos:

1. Se eligen **K centroides al azar**.
2. Se forman los clústeres asignando cada punto a su centroide más cercano.
3. Se calcula la media de cada clúster y **el centroide se mueve a esa media**.
4. Se repiten los pasos 2 y 3 hasta cumplir un criterio de parada — normalmente, cuando los centroides ya apenas se mueven y los clústeres se vuelven estáticos.
5. Cuando los clústeres dejan de cambiar, el algoritmo ha **convergido**.

> **Punto importante**: como el punto de partida de los centroides es **aleatorio**, volver a ejecutar el algoritmo puede dar clústeres ligeramente distintos. Por eso el entrenamiento normalmente hace **varias iteraciones**, reinicializando los centroides cada vez, y se queda con el modelo de mejor WCSS. Ese es justamente el parámetro `n_init`.

In [ ]:
from sklearn.cluster import KMeans

# n_init=100 → reinicia 100 veces y se queda con el mejor resultado
# random_state fija la semilla para que este cuaderno sea reproducible
model = KMeans(n_clusters=3, init='k-means++', n_init=100, max_iter=1000, random_state=0)

km_clusters = model.fit_predict(features.values)

print('Asignación de clúster de cada observación:')
print(km_clusters)

In [ ]:
def plot_clusters(samples, clusters, titulo='Asignaciones'):
    """Dibuja los puntos en 2D coloreados por el clúster al que pertenecen."""
    col_dic = {0:'blue', 1:'green', 2:'orange'}
    mrk_dic = {0:'*', 1:'x', 2:'+'}
    plt.figure(figsize=(7,6))
    for c in np.unique(clusters):
        mask = clusters == c
        plt.scatter(samples[mask,0], samples[mask,1],
                    color=col_dic[c], marker=mrk_dic[c], s=100, label=f'Clúster {c}')
    plt.xlabel('Dimensión 1')
    plt.ylabel('Dimensión 2')
    plt.title(titulo)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_clusters(features_2d, km_clusters, 'K-Means (k=3)')

## Evaluar la separación: coeficiente de silueta

Para cuantificar qué tan bien separados quedaron los clústeres se usa el **coeficiente de silueta**: un valor entre -1 y +1. Cuanto más cerca de +1, mejor separados están.

In [ ]:
from sklearn.metrics import silhouette_score

score_km = silhouette_score(features.values, km_clusters)
print(f'Silueta (K-Means): {score_km:.3f}')

## ¿Para qué sirve esto en la práctica?

A veces tienes datos que necesitas agrupar sin saber cuántos grupos hay ni qué significan. Por ejemplo, una organización de marketing puede querer separar a sus clientes en segmentos y luego investigar cómo se comporta cada uno.

También se usa como **paso previo a una clasificación**: primero identificas grupos, luego les asignas etiquetas de clase a mano, y con esos datos ya etiquetados entrenas un clasificador.

En el caso de las semillas, las especies **ya se conocen** (0 = Kama, 1 = Rosa, 2 = Canadian), así que podemos comparar las especies reales contra los clústeres que encontró el algoritmo **sin haberlas visto nunca**.

In [ ]:
seed_species = data[data.columns[7]]
plot_clusters(features_2d, seed_species.values, 'Especies reales (etiquetas conocidas)')

Hay algunas diferencias entre las asignaciones de clúster y las etiquetas reales, pero K-Means hizo un trabajo razonable: las semillas de la misma especie caen mayormente en el mismo clúster.

**Esto es notable**: el algoritmo no vio ni una sola etiqueta y aun así reconstruyó aproximadamente la estructura real de especies, solo a partir de las medidas físicas.

## Agrupación jerárquica

Los métodos jerárquicos hacen menos suposiciones sobre la distribución de los datos que K-Means. A cambio, K-Means suele ser bastante más escalable.

La agrupación jerárquica construye clústeres por dos vías:

- **Divisiva** — enfoque "de arriba abajo": empieza con todo el dataset y va encontrando particiones paso a paso.
- **Aglomerativa** — enfoque "de abajo arriba", que es el que usaremos:
  1. Se calculan las distancias de enlace entre todos los puntos.
  2. Los puntos se agrupan por pares con su vecino más cercano.
  3. Se calculan las distancias de enlace entre los clústeres.
  4. Los clústeres se combinan por pares en clústeres mayores.
  5. Se repiten los pasos 3 y 4 hasta que todos los puntos están en un único clúster.

### Función de enlace

Cómo se mide la distancia *entre clústeres*:

| Enlace | Qué mide |
|---|---|
| **Ward** | El aumento de varianza al unir los dos clústeres |
| **Average** (promedio) | La distancia media entre todos los pares de miembros |
| **Complete** (máximo) | La distancia máxima entre miembros de los dos clústeres |

### Métrica de distancia

Cómo se mide la distancia *entre puntos*:

| Métrica | Notas |
|---|---|
| **Euclídea** (l2) | La más usada. Es la **única** opción válida con enlace Ward |
| **Manhattan** (l1) | Robusta frente a valores atípicos |
| **Similitud del coseno** | Producto escalar de los vectores dividido por sus magnitudes. Ojo: mide **similitud**, mientras que las otras dos miden **diferencia**. Útil con datos como imágenes o texto |

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agg_model = AgglomerativeClustering(n_clusters=3)
agg_clusters = agg_model.fit_predict(features.values)

print('Asignación de clúster de cada observación:')
print(agg_clusters)

In [ ]:
plot_clusters(features_2d, agg_clusters, 'Agrupación jerárquica aglomerativa (k=3)')

In [ ]:
score_agg = silhouette_score(features.values, agg_clusters)
print(f'Silueta (jerárquica): {score_agg:.3f}')

## Comparativa

In [ ]:
comparativa = pd.DataFrame([
    {'algoritmo': 'K-Means', 'silueta': score_km},
    {'algoritmo': 'Jerárquica aglomerativa', 'silueta': score_agg},
]).sort_values('silueta', ascending=False).reset_index(drop=True)

comparativa.round(4)

## Resumen

Practicamos K-Means y agrupación jerárquica. Este aprendizaje no supervisado es capaz de tomar datos **sin etiquetar** e identificar qué observaciones se parecen entre sí.

Cuándo elegir cada uno:

| | K-Means | Jerárquica |
|---|---|---|
| ¿Hay que fijar el número de clústeres? | Sí, `k` es obligatorio | No necesariamente |
| Escalabilidad | Buena, incluso con muchos datos | Peor; se complica con datasets grandes |
| Suposiciones sobre la distribución | Más | Menos |
| Interpretabilidad | Grupos planos | Muestra además **relaciones entre grupos** |